In [ ]:
import pandas as pd
import datetime as dt
import numpy as np
import math
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report, roc_curve, roc_auc_score


from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression, SGDClassifier, BayesianRidge
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, GradientBoostingRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVR, SVC
# from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost.sklearn import XGBClassifier

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# sns.set_theme()
sns.set_palette('rocket_r')
# sns.color_palette("rocket")

## To do:
1. Preprocess:
   
   a. Standardize
   
   b. Split Data
   
   c. Normalize
3. Extract import features using RandomForestClassifier
4. Run all models storing their accuracy and f1-scores in a table

In [ ]:
data = pd.read_csv('../../data/model_features.csv')
data.shape

In [ ]:
data.head()

In [ ]:
models = {
    'Bayesian Ridge Regression':[BayesianRidge(), False, 'regression', 'Linear'],
    'Decision Tree Classifier':[DecisionTreeClassifier(random_state=42), False, 'classification', 'Tree'],
    'Gaussian Na?ve Bayes':[GaussianNB(), False, 'classification', 'Probabilistic'],
    'Gradient Boosting for Classification':[GradientBoostingClassifier(random_state=42), False, 'classification', 'Tree'],
    'Gradient Boosting for Regression':[GradientBoostingRegressor(random_state=42), False, 'regression', 'Tree'],
    'Histogram-based Gradient Boosting Classification Tree':[HistGradientBoostingClassifier(random_state=42), True, 'classification', 'Tree'],
    'Logistic Regression':[LogisticRegression(random_state=42), False, 'classification', 'Linear'],
    'Random Forest Classifier':[RandomForestClassifier(random_state=42), False, 'classification', 'Tree'],
    'Stochastic Gradient Descent Classifier':[SGDClassifier(random_state=42), False, 'classification', 'Linear'],
    'Epsilon-Support Vector Regression (SVR)':[SVR(), False, 'regression', 'Non-Probabilistic'],
    'Support Vector Machines Classifier':[SVC(random_state=42), False, 'classification', 'Non-Probabilistic'],
    'Extreme Gradient Boosting (Xgboost) Classifier':[XGBClassifier(random_state=42), False, 'classification', 'Tree']
}

In [ ]:
# custom function
def sigmoid(x):
  return 1 / (1 + math.exp(-x))

In [ ]:
def transform_to_binary(x):
    if x < 0:
        return 0
    elif x > 1:
        return 1
    else:
        return round(x)

In [ ]:
results = {}
for model_type in models.keys():
    start = dt.datetime.now()
    # Remove fields with NaN values if model can't handel them
    if not models[model_type][1]:
        df = data.dropna(axis=1).copy()
    else:
        df = data.copy()

    # Split training and test data
    X = df.iloc[:,1:]
    y = df.iloc[:,0]

    # Initialize the StratifiedKFold with 5 folds
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # # Standardize/Normalize the data
    # if models[model_type][2] == 'regression':
    #     scaler = MinMaxScaler() #normalization
    # else:
    #     scaler = StandardScaler() #standardization

    model = models[model_type][0]
    
    # Perform 5-fold cross-validation
    f1_scores = []
    accuracies = []
    for train_index, test_index in kfold.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
        # Standardize and normalize the data using the same scaler
        for scaler in [StandardScaler(), MinMaxScaler()]:
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
    
        # Train the model on the training set
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        if models[model_type][2] == 'regression':
            # define vectorized sigmoid
            sigmoid_v = np.vectorize(sigmoid)
            y_pred_sig = sigmoid_v(np.array(y_pred))
            y_pred = list(map(round,y_pred_sig))

        f1_scores.append(f1_score(y_test, y_pred))
        accuracies.append(accuracy_score(y_test, y_pred))

    f1_score_strat = sum(f1_scores) / len(f1_scores)
    f1_std = np.std(f1_scores)
    accuracy_score_strat = sum(accuracies) / len(accuracies)
    acc_std = np.std(accuracies)

    runtime = (dt.datetime.now() - start).seconds
    results[model_type] = [models[model_type][3], f1_score_strat,f1_std, accuracy_score_strat,acc_std,runtime]
    print(f'{model_type}\nAccuracy: {accuracy_score_strat}\nF1-Score: {f1_score_strat}\nRuntime: {runtime}s\n')

In [ ]:
df_results = pd.DataFrame(results).T
df_results.columns = ['model_type','f1_score','f1_score_?','accuracy_score','accuracy_?','runtime']
df_results.sort_values('f1_score', ascending=False).to_csv('../../data/model_summary_phase1.csv')

In [ ]:
df_results = df_results.sort_values('f1_score', ascending=False)
df_results

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=df_results, y=df_results.index, x='f1_score', hue='model_type')

# plt.title('Figure 1 - Attempted Models Ranked by F1-score')
plt.ylabel(None)
plt.xlabel('F1-score')
plt.legend(title='Model Type')
plt.savefig('../../visuals/models_ranked_by_f1_score.png', bbox_inches='tight')

## Identify important features using Random Forest Classifier

In [ ]:
# Split training and test data
df = data.fillna(0)
# df = data.copy()
X = df.iloc[:,1:]
y = df.iloc[:,0]

rf_model = RandomForestClassifier()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for scaler in [StandardScaler(), MinMaxScaler()]:
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

rf_model.fit(X_train, y_train)

In [ ]:
n = 10
feature_importance = {key:value for key,value in sorted(zip(data.columns.tolist()[1:], rf_model.feature_importances_), key=lambda x: x[1], reverse=True)}
top_features = list(feature_importance.keys())[:n]

In [ ]:
feature_importance

In [ ]:
top_features

## Run models only using top features

In [ ]:
results = {}
data1 = data[['won']+top_features]
for model_type in models.keys():
    start = dt.datetime.now()
    # Remove fields with NaN values if model can't handel them
    if not models[model_type][1]:
        df = data1.dropna(axis=1).copy()
    else:
        df = data1.copy()

    # Split training and test data
    X = df.iloc[:,1:]
    y = df.iloc[:,0]

    # Initialize the StratifiedKFold with 5 folds
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # # Standardize/Normalize the data
    # if models[model_type][2] == 'regression':
    #     scaler = MinMaxScaler() #normalization
    # else:
    #     scaler = StandardScaler() #standardization

    model = models[model_type][0]
    
    # Perform 5-fold cross-validation
    f1_scores = []
    accuracies = []
    for train_index, test_index in kfold.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
        # Standardize and normalize the data using the same scaler
        for scaler in [StandardScaler(), MinMaxScaler()]:
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
    
        # Train the model on the training set
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        if models[model_type][2] == 'regression':
            # define vectorized sigmoid
            sigmoid_v = np.vectorize(sigmoid)
            y_pred_sig = sigmoid_v(np.array(y_pred))
            y_pred = list(map(round,y_pred_sig))

        f1_scores.append(f1_score(y_test, y_pred))
        accuracies.append(accuracy_score(y_test, y_pred))

    f1_score_strat = sum(f1_scores) / len(f1_scores)
    f1_std = np.std(f1_scores)
    accuracy_score_strat = sum(accuracies) / len(accuracies)
    acc_std = np.std(accuracies)

    runtime = (dt.datetime.now() - start).seconds
    results[model_type] = [models[model_type][3], f1_score_strat,f1_std, accuracy_score_strat,acc_std,runtime]
    print(f'{model_type}\nAccuracy: {accuracy_score_strat}\nF1-Score: {f1_score_strat}\nRuntime: {runtime}s\n')

In [ ]:
df_results = pd.DataFrame(results).T
df_results.columns = ['model_type','f1_score','f1_score_?','accuracy_score','accuracy_?','runtime']
df_results.sort_values('f1_score', ascending=False).to_csv('../../data/model_summary_phase2.csv')
df_results.sort_values('f1_score', ascending=False)

In [ ]:
df_results_total = pd.read_csv('../../data/model_summary_phase1.csv').rename(columns={'Unnamed: 0':'model'})
df_results_top10 = pd.read_csv('../../data/model_summary_phase2.csv').rename(columns={'Unnamed: 0':'model'})

In [ ]:
df_results_top10['model'] = df_results_top10['model'].apply(lambda x:x+'_10')

In [ ]:
combined_results = pd.concat([df_results_total,df_results_top10]).set_index('model').sort_values('f1_score', ascending=False)
combined_results

In [ ]:
plt.figure(figsize=(15,6))
sns.barplot(data=combined_results, y=combined_results.index, x='f1_score', hue='model_type')

plt.title('Figure 1 - Attempted Models Ranked by F1-score')
plt.ylabel('Model')
plt.xlabel('F1-score')
plt.legend(title='Model Type')

In [ ]:
plt.figure(figsize=(15,6))
combined_results_runtime = combined_results.sort_values('runtime',ascending=False)
sns.barplot(data=combined_results_runtime, y=combined_results_runtime.index, x='runtime', hue='model_type')

In [ ]:
sns.scatterplot(data=combined_results,x='runtime',y='f1_score',hue='model_type',)

## Tune Top Three Models:

- Histogram-based Gradient Boosting Classification Tree
- Logistic Regression
- Stochastic Gradient Descent Classifier

In [ ]:
models_to_tune = {
    'Histogram-based Gradient Boosting Classification Tree':[HistGradientBoostingClassifier(), True, 'classification', 'Tree'],
    'Logistic Regression':[LogisticRegression(), False, 'classification', 'Linear'],
    'Stochastic Gradient Descent Classifier':[SGDClassifier(), False, 'classification', 'Linear'],
}

In [ ]:
tuned_results = {}

## Tune Histogram-based Gradient Boosting Classification Tree

In [ ]:
model_name = 'Best Histogram-based Gradient Boosting Classification Tree'
start = dt.datetime.now()
df = data.copy()

X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for scaler in [StandardScaler(), MinMaxScaler()]:
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)


hgb_classifier = HistGradientBoostingClassifier(random_state=42)

param_grid = {
    'learning_rate': [x/100 for x in list(range(2,9))],
    'max_iter': [100],
    'max_depth': [None, 5, 10],
    'min_samples_leaf': [15, 20, 25]
}

# Use StratifiedKFold for cross-validation
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hgb_grid_search = GridSearchCV(hgb_classifier, param_grid, cv=stratified_cv, scoring='f1')
hgb_grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", hgb_grid_search.best_params_)

# Get the best model from the grid search
best_hgb_model = hgb_grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_hgb_model.predict(X_test)

# Evaluate the accuracy of the model
f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

runtime = (dt.datetime.now() - start).seconds
print(model_name)
print('CV Training Best F1-Score:', hgb_grid_search.best_score_)
print("Test F1-Score:", f1)
print("Test Accuracy:", accuracy)
print('Tuning Runtime:', runtime, 'sec')

parameters_str = ', '.join([f'{key}: {value}' for key,value in hgb_grid_search.best_params_.items()])
tuned_results[model_name] = [f1,accuracy,parameters_str, runtime]

#### Try without scaling

In [ ]:
model_name = 'Best Histogram-based Gradient Boosting Classification Tree without Scaling'
start = dt.datetime.now()
df = data.copy()

X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


hgb_classifier = HistGradientBoostingClassifier(random_state=42)

param_grid = {
    'learning_rate': [x/100 for x in list(range(2,9))],
    'max_iter': [100],
    'max_depth': [None, 5, 10],
    'min_samples_leaf': [15, 20, 25]
}

# Use StratifiedKFold for cross-validation
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hgb_grid_search = GridSearchCV(hgb_classifier, param_grid, cv=stratified_cv, scoring='f1')
hgb_grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", hgb_grid_search.best_params_)

# Get the best model from the grid search
best_hgb_model = hgb_grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_hgb_model.predict(X_test)

# Evaluate the accuracy of the model
f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

runtime = (dt.datetime.now() - start).seconds
print(model_name)
print('CV Training Best F1-Score:', hgb_grid_search.best_score_)
print("Test F1-Score:", f1)
print("Test Accuracy:", accuracy)
print('Tuning Runtime:', runtime, 'sec')

parameters_str = ', '.join([f'{key}: {value}' for key,value in hgb_grid_search.best_params_.items()])
tuned_results[model_name] = [f1,accuracy,parameters_str, runtime]

In [ ]:
data.columns

#### Try with differenced columns:

In [ ]:
data1 = data.copy()
drop_cols = []
for col in data.columns[1:].tolist():
    if col+'_opp' in data.columns:
        drop_cols.append(col)
        drop_cols.append(col+'_opp')
        data1[col+'_diff'] = data1[col] - data1[col+'_opp']
    
data1.drop(drop_cols, axis=1, inplace=True)

In [ ]:
data1.columns

In [ ]:
model_name = 'Best Histogram-based Gradient Boosting Classification Tree - team_opp_metric_difference'
start = dt.datetime.now()
df = data1.copy()

X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# for scaler in [StandardScaler(), MinMaxScaler()]:
#     X_train = scaler.fit_transform(X_train)
#     X_test = scaler.transform(X_test)


hgb_classifier = HistGradientBoostingClassifier(random_state=42)

param_grid = {
    'learning_rate': [x/100 for x in list(range(2,9))],
    'max_iter': [100],
    'max_depth': [None, 5, 10],
    'min_samples_leaf': [15, 20, 25]
}

# Use StratifiedKFold for cross-validation
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hgb_grid_search = GridSearchCV(hgb_classifier, param_grid, cv=stratified_cv, scoring='f1')
hgb_grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", hgb_grid_search.best_params_)

# Get the best model from the grid search
best_hgb_model = hgb_grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_hgb_model.predict(X_test)

# Evaluate the accuracy of the model
f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

runtime = (dt.datetime.now() - start).seconds
print(model_name)
print('CV Training Best F1-Score:', hgb_grid_search.best_score_)
print("Test F1-Score:", f1)
print("Test Accuracy:", accuracy)
print('Tuning Runtime:', runtime, 'sec')

parameters_str = ', '.join([f'{key}: {value}' for key,value in hgb_grid_search.best_params_.items()])
tuned_results[model_name] = [f1,accuracy,parameters_str, runtime]

## Tune Logistic Regression

In [ ]:
warnings.filterwarnings("ignore")
model_name = 'Best Logistic Regression'
start = dt.datetime.now()
df = data.dropna(axis=1).copy()

X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for scaler in [StandardScaler(), MinMaxScaler()]:
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)


lg_classifier = LogisticRegression(random_state=42)

param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['lbfgs','liblinear', 'saga'],
    'max_iter':[50,100,1000]
}

# Use StratifiedKFold for cross-validation
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lg_grid_search = GridSearchCV(lg_classifier, param_grid, cv=stratified_cv, scoring='f1')
lg_grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", lg_grid_search.best_params_)

# Get the best model from the grid search
best_lg_model = lg_grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_lg_model.predict(X_test)

# Evaluate the accuracy of the model
f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

runtime = (dt.datetime.now() - start).seconds
print(model_name)
print('CV Training Best F1-Score:', lg_grid_search.best_score_)
print("Test F1-Score:", f1)
print("Test Accuracy:", accuracy)
print('Tuning Runtime:', runtime, 'sec')

parameters_str = ', '.join([f'{key}: {value}' for key,value in lg_grid_search.best_params_.items()])
tuned_results[model_name] = [f1,accuracy,parameters_str, runtime]

## Tune Stochastic Gradient Descent

In [ ]:
warnings.filterwarnings("always")
model_name = 'Best Stochastic Gradient Descent'
start = dt.datetime.now()
df = data.dropna(axis=1).copy()

X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for scaler in [StandardScaler(), MinMaxScaler()]:
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)


sgd_classifier = SGDClassifier(random_state=42)

param_grid = {
    'alpha': [0.00005, 0.0001, 0.001],
    'penalty': ['l1', 'l2'],
    'loss': ['hinge', 'log', 'modified_huber'],
    'max_iter':[500,1000,1500]
}

# Use StratifiedKFold for cross-validation
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

sgd_grid_search = GridSearchCV(sgd_classifier, param_grid, cv=stratified_cv, scoring='f1')
sgd_grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", sgd_grid_search.best_params_)

# Get the best model from the grid search
best_sgd_model = sgd_grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_sgd_model.predict(X_test)

# Evaluate the accuracy of the model
f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

runtime = (dt.datetime.now() - start).seconds
print(model_name)
print('CV Training Best F1-Score:', sgd_grid_search.best_score_)
print("Test F1-Score:", f1)
print("Test Accuracy:", accuracy)
print('Tuning Runtime:', runtime, 'sec')

parameters_str = ', '.join([f'{key}: {value}' for key,value in sgd_grid_search.best_params_.items()])
tuned_results[model_name] = [f1,accuracy,parameters_str, runtime]

In [ ]:
best_models = pd.DataFrame(tuned_results).T
best_models.columns = ['f1_score','accuracy_score','parameters','cv_runtime(sec)']
best_models.to_csv('../../data/best_models.csv')
best_models